In [1]:
from openai import OpenAI
from IPython.display import display, Markdown

In [2]:
client = OpenAI(
    base_url="YOUR-BASE-URL",
    api_key="YOUR-API-KEY"
)

In [3]:
chat = client.chat.completions.create(
    messages=[
        {
            "role" : "system",
            "content" : "you are a helpful ai assistant.your name is 'Jarvis'"
        },
        {
            "role"  : "user",
            "content" : "The capital of India?"
        }
    ],
    model="openai/gpt-oss-120b",
    temperature=0,
    reasoning_effort="high"
)

response = chat.choices[0]
bot = response.message.content
reasoning  = response.message.reasoning

display(Markdown(f"**BOT:**\n\n{bot}"))
display(Markdown(f"**REASONING:**\n\n{reasoning}"))

**BOT:**

The capital of India is **New Delhi** – a district within the larger National Capital Territory of Delhi.

**REASONING:**

The user asks: "The capital of India?" This is a straightforward question. The answer: New Delhi. Possibly also mention that New Delhi is a district within the larger city of Delhi, which is the National Capital Territory. Provide a concise answer. The user might want just the capital. So answer: New Delhi. Could also add some context. The user didn't ask for anything else. So answer succinctly.

But the instruction: "You are a helpful AI assistant. Your name is 'Jarvis'." So we should respond as Jarvis. Possibly sign off with name? Not required. Provide answer.

Thus answer: The capital of India is New Delhi. Also mention that it's part of the National Capital Territory of Delhi. Provide a short explanation.

Thus final answer.

In [37]:
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import classification_report

In [5]:
def chatgpt_generation(prompt, document, model="openai/gpt-oss-120b"):
  messages =[
      {"role" : "system",
       "content" : "you are a helpful ai assistant.your name is 'Jarvis'"},
        {"role"  : "user",
         "content" : prompt.replace("[DOCUMENT]", document)}
      ]

  chat_completion = client.chat.completions.create(
      messages= messages,
      model = model,
      temperature=0,
      reasoning_effort = "high"
  )
  return {"classification": chat_completion.choices[0].message.content,
          "reasoning" : chat_completion.choices[0].message.reasoning }

In [ ]:
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

In [6]:
#Example
document = "The service was terrible."
chatgpt_generation(prompt, document)

{'classification': '0',
 'reasoning': 'The user asks: "Predict whether the following document is a positive or negative movie review: The service was terrible. If it is positive return 1 and if it is negative return 0. Do not give any other answers."\n\nWe need to output either 1 or 0. The statement "The service was terrible." is negative sentiment. So output 0.\n\nWe must not give any other answers. So just "0".'}

In [ ]:
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")

In [38]:
idx = np.random.randint(0, len(data['test']['label']), size=50)

prediction = [
    chatgpt_generation(prompt, doc)
    for doc in tqdm(data['test']['text'][idx])
]

y_pred = [int(pred['classification']) for pred in prediction]

100%|██████████| 50/50 [01:02<00:00,  1.25s/it]


In [39]:
def evaluate(y_true, y_pred):
  performance = classification_report(y_true, y_pred, target_names=['Negative Review', "Positive Review"], labels = [1,0])
  print(performance)


In [41]:
evaluate(data['test']['label'][idx], y_pred)

                 precision    recall  f1-score   support

Negative Review       1.00      0.95      0.98        22
Positive Review       0.97      1.00      0.98        28

       accuracy                           0.98        50
      macro avg       0.98      0.98      0.98        50
   weighted avg       0.98      0.98      0.98        50



In [43]:
print(data['test']['label'][idx])
print(y_pred)

[0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1]
[0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1]
